# Formative 2 — Voice pipeline (Colab)

**Runtime → Run all.** You only upload **6 WAV files** when asked.

| File |
|------|
| `beulla_yes_approve.wav` |
| `beulla_confirm_transaction.wav` |
| `ibrahim_yes_approve.wav` |
| `ibrahim_confirm_transaction.wav` |
| `kelvin_yes_approve.wav` |
| `kelvin_confirm_transaction.wav` |

In [ ]:
%pip install -q librosa soundfile pandas matplotlib scikit-learn joblib

In [ ]:
from pathlib import Path
from google.colab import files

ROOT = Path("/content/voice_project")
RAW = ROOT / "data" / "audio" / "raw"
AUG = ROOT / "data" / "audio" / "augmented"
FIG = ROOT / "outputs" / "figures" / "audio"
MODELS = ROOT / "models"
for d in (RAW, AUG, FIG, MODELS):
    d.mkdir(parents=True, exist_ok=True)

print("Select all 6 WAV files:")
for name, data in files.upload().items():
    (RAW / name.lower()).write_bytes(data)
    print(" ", name.lower())

wavs = sorted(RAW.glob("*.wav"))
assert len(wavs) == 6, f"Need 6 WAVs, got {len(wavs)}: {[w.name for w in wavs]}"

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from IPython.display import display, Image, Audio

PHRASES = ("yes_approve", "confirm_transaction")
SR = 22_050
N_MFCC = 13

def parse_name(path: Path):
    stem = path.stem.lower()
    for phrase in PHRASES:
        if f"_{phrase}" in stem:
            return stem.split(f"_{phrase}")[0], phrase
    raise ValueError(f"Bad filename: {path.name}")

def feat_cols():
    c = [f"mfcc_{i}_mean" for i in range(N_MFCC)]
    c += [f"mfcc_{i}_std" for i in range(N_MFCC)]
    c += ["spectral_rolloff_mean", "spectral_rolloff_std", "energy_mean", "energy_std", "duration_sec"]
    return c

def extract_features(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    rms = librosa.feature.rms(y=y)
    out = {}
    for i in range(N_MFCC):
        out[f"mfcc_{i}_mean"] = float(np.mean(mfcc[i]))
        out[f"mfcc_{i}_std"] = float(np.std(mfcc[i]))
    out["spectral_rolloff_mean"] = float(np.mean(rolloff))
    out["spectral_rolloff_std"] = float(np.std(rolloff))
    out["energy_mean"] = float(np.mean(rms))
    out["energy_std"] = float(np.std(rms))
    out["duration_sec"] = float(len(y) / sr)
    return out

def plot_wave_spec(y, sr, title):
    fig, axes = plt.subplots(2, 1, figsize=(10, 5))
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Waveform — {title}")
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="hz", ax=axes[1])
    axes[1].set_title(f"Spectrogram — {title}")
    plt.tight_layout()
    plt.show()

AUGS = [
    ("pitch_up", lambda y: librosa.effects.pitch_shift(y, sr=SR, n_steps=2)),
    ("time_stretch", lambda y: librosa.effects.time_stretch(y, rate=1.12)),
]

def verify_file(wav_path: Path, clf, le, threshold=0.5):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    feats = extract_features(y, SR)
    row = np.array([[feats[c] for c in feat_cols()]])
    proba = clf.predict_proba(row)[0]
    idx = int(np.argmax(proba))
    member = le.classes_[idx]
    conf = float(proba[idx])
    ok = conf >= threshold
    print(f"  Predicted: {member} (confidence={conf:.3f}) → {'APPROVED' if ok else 'DENIED'}")
    return ok

In [ ]:
# --- Process: augment + features + plots ---
rows = []
for path in sorted(RAW.glob("*.wav")):
    member, phrase = parse_name(path)
    base = f"{member}_{phrase}"
    y, _ = librosa.load(path, sr=SR, mono=True)
    print(f"\n=== {path.name} ===")
    display(Audio(y, rate=SR))
    plot_wave_spec(y, SR, f"{base} (original)")
    rows.append({"member_id": member, "phrase": phrase, "augmentation": "original",
                 "file_path": path.name, **extract_features(y, SR)})
    for aname, fn in AUGS:
        y_a = fn(y)
        sf.write(AUG / f"{base}_{aname}.wav", y_a, SR)
        plot_wave_spec(y_a, SR, f"{base} ({aname})")
        rows.append({"member_id": member, "phrase": phrase, "augmentation": aname,
                     "file_path": f"{base}_{aname}.wav", **extract_features(y_a, SR)})

cols = ["member_id", "phrase", "augmentation", "file_path"] + feat_cols()
df = pd.DataFrame(rows)[cols]
df.to_csv(ROOT / "audio_features.csv", index=False)
display(df)
print("Rows:", len(df))

In [ ]:
# --- Train voiceprint model ---
X = df[feat_cols()].values
y = df["member_id"].values
le = LabelEncoder()
y_enc = le.fit_transform(y)
Xt, Xv, yt, yv = train_test_split(X, y_enc, test_size=0.25, random_state=42, stratify=y_enc)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xt, yt)
pred = clf.predict(Xv)
proba = clf.predict_proba(Xv)
metrics = {
    "accuracy": float(accuracy_score(yv, pred)),
    "f1_weighted": float(f1_score(yv, pred, average="weighted")),
    "loss": float(log_loss(yv, proba)),
    "n_samples": int(len(df)),
    "members": list(le.classes_),
}
print(json.dumps(metrics, indent=2))
joblib.dump({"model": clf, "label_encoder": le, "feature_columns": feat_cols()}, MODELS / "voice_verifier.joblib")

In [ ]:
# --- EDA ---
fig, ax = plt.subplots(figsize=(8, 4))
df.groupby("member_id")["energy_mean"].mean().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Mean RMS energy by member")
plt.show()

mfcc_cols = [c for c in df.columns if c.startswith("mfcc_") and c.endswith("_mean")]
corr = df[mfcc_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(mfcc_cols)), mfcc_cols, rotation=45, ha="right")
ax.set_yticks(range(len(mfcc_cols)), mfcc_cols)
ax.set_title("MFCC mean correlations")
fig.colorbar(im)
plt.show()

In [ ]:
# --- System demo (voice gate; face/product simulated) ---
print("=== FULL TRANSACTION (simulated) ===")
print("1) Face: APPROVED (teammate model)")
print("2) Voice:")
verify_file(RAW / "beulla_yes_approve.wav", clf, le)
print("3) Product model: prediction shown (teammate)")

print("\n=== UNAUTHORIZED ===")
print("1) Face: denied or wrong user (simulated)")
noise_path = RAW / "_unauthorized_noise.wav"
sf.write(noise_path, np.random.randn(SR * 2).astype(np.float32) * 0.05, SR)
print("2) Voice:")
verify_file(noise_path, clf, le)

In [ ]:
import zipfile
from google.colab import files

out_zip = Path("/content/voice_deliverables.zip")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT / "audio_features.csv", "audio_features.csv")
    z.write(MODELS / "voice_verifier.joblib", "models/voice_verifier.joblib")
    z.writestr("voice_model_metrics.json", json.dumps(metrics, indent=2))
    for p in AUG.glob("*.wav"):
        z.write(p, f"augmented/{p.name}")
print("Download deliverables for GitHub:")
files.download(str(out_zip))